<a href="https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adeleke1508/flyrank_week_one_ml_project_1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Task Type: Clustering
The goal isn't to predict a single known outcome (like "will this page decline") — it's to discover what natural types of pages exist across a 500K+ item inventory that no one has explicitly labeled yet. Clustering is the right tool because it doesn't require a target variable; it finds structure directly from how pages behave across multiple signals at once (traffic, position, engagement, freshness, age). That's exactly the kind of question — "what recurring patterns exist that we haven't named yet" — that clustering is built for, versus classification (which needs a known label to predict) or ranking/scoring (which needs a defined target to rank against). Since FlyRank's inventory is too large to review page-by-page, clustering turns an unmanageable list into a small number of interpretable groups a reviewer can reason about and act on in batches.


Action the Output Support: Instead of a Reviewer loking at 500k+ pages one at a time , they open a cluster and treat the group with one action - protect, refresh, prune, or just monitor - because pages in the same cluster behave alike.



In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There isn't a fixed Target, The 'proxy' is the combination of features clustered together. The output is a cluster label per page , which are then interpreted and named

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

*Silhouette score - Do the cluster actually separate cleanly or overlap into mush?
*Stability - do you get roughly the same groups if you re-run with a different seed or slightly different feature set?
*Human-inspection validity - when you actually read 10-15 real pages from each clusters, do they make intuitive sense as a 'type'?


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [3]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [4]:
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df = df[(df['impressions_90d'] > 0 ) & (df['content_age_days'] >= 90)]
df = df.drop_duplicates(subset= 'content_id')
print(f"Rows (= pages) in my lane's slice: {len(df)}")

Rows (= pages) in my lane's slice: 30000


In [6]:
unit_cols = ['content_id', 'client_id', 'impressions_90d', 'clicks_90d',
             'sessions_90d', 'content_age_days', 'word_count', 'avg_position',
             'ctr', 'engagement_rate', 'scroll_rate']
display_df = df[unit_cols].head()
print('\nUnit of analysis - one row = one content item (page):')
print(display_df)


Unit of analysis - one row = one content item (page):
             content_id          client_id  impressions_90d  clicks_90d  \
0  content_304f48230142  client_f369cb89fc             3803          29   
1  content_a1fb4e703a9e  client_4e07408562            15320           7   
2  content_9aa793d4d895  client_7f2253d7e2            12581          11   
3  content_331d6c4de07b  client_19581e27de            11751          58   
4  content_d99b7a2d90ca  client_3fdba35f04            19140          24   

   sessions_90d  content_age_days  word_count  avg_position   ctr  \
0            17               187      3221.0          10.6  0.76   
1             9               445      2481.0          20.3  0.05   
2            11               141      3515.0          36.5  0.09   
3            78               463         NaN           6.2  0.49   
4           145               263      2803.0          44.0  0.13   

   engagement_rate  scroll_rate  
0             5.88         4.55  
1          

In [7]:
#Numeric feature to cluster on
numeric_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'content_age_days',
                'word_count', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate']
X = df[numeric_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
X_scaled = StandardScaler().fit_transform(X)

In [8]:
#PCA: is there real structure ?
pca = PCA(n_components=2)
pca.fit(X_scaled)
print(f'\nVariance Explained by 2 PCA Components: {pca.explained_variance_ratio_.sum():.1%}')


Variance Explained by 2 PCA Components: 44.3%


In [9]:
#KMeans: Sketching what the Proxy Columns(Cluster Label) look like ---
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster_label'] = kmeans.fit_predict(X_scaled)

sil = silhouette_score(X_scaled, df['cluster_label'])
print(f'Silhouette score (5 clusters): {sil:.3f}')
print(f'Cluster sizes: \n{df['cluster_label'].value_counts().sort_index()}')

print('\nSketch of the target column, sitting next to the unit of analysis:')
print(df[unit_cols + ['cluster_label']].head(8))

Silhouette score (5 clusters): 0.279
Cluster sizes: 
cluster_label
0      611
1     3394
2    13816
3    11879
4      300
Name: count, dtype: int64

Sketch of the target column, sitting next to the unit of analysis:
             content_id          client_id  impressions_90d  clicks_90d  \
0  content_304f48230142  client_f369cb89fc             3803          29   
1  content_a1fb4e703a9e  client_4e07408562            15320           7   
2  content_9aa793d4d895  client_7f2253d7e2            12581          11   
3  content_331d6c4de07b  client_19581e27de            11751          58   
4  content_d99b7a2d90ca  client_3fdba35f04            19140          24   
5  content_d4084a4bc775  client_f369cb89fc             3970           1   
6  content_9a34b442b552  client_8722616204               20           0   
7  content_a63219c6e95a  client_19581e27de             1724           1   

   sessions_90d  content_age_days  word_count  avg_position   ctr  \
0            17               187      

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can only reasom about one or two conditions at a time and someone has to handpick which conditions matter. But content behavior is shaped by many signals interacting as one - position, freshness, engagement, and volume all moving together in a way human wouldn't manually encode into if/else logic. Clustering lets the natural groupings emerge directly from the data own structure potentially surfacing archetypes that nobody thought to write a rule for in advance- because we do not the the categories that exist until the data reveals them.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.